In [ ]:
import os
from pathlib import Path
import numpy as np
import pandas as pd
import scipy.stats as stats
import matplotlib.pyplot as plt
import seaborn as sns

# Setup root directory paths
ROOT = Path("D:/Bussiness_plan/Multimodal_PM25")
OUTPUT_DIR = ROOT / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Thiết lập phong cách hiển thị hình vẽ (Aesthetics)
sns.set_theme(style="whitegrid")
plt.rcParams.update({
    "font.size": 11,
    "axes.labelsize": 12,
    "axes.titlesize": 13,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    "figure.titlesize": 15,
    "legend.fontsize": 10,
    "figure.dpi": 200
})

def calculate_stats_and_data():
    # 1. Load Ground Observations (PM2.5) từ OpenAQ
    df_aq = pd.read_csv(ROOT / "data/interim/openaq/all_stations_daily.csv")
    
    # PM2.5 thô (loại trừ các dòng được nội suy)
    pm25_raw = df_aq[df_aq["pm25_was_interpolated"] == 0]["pm25_mean"].dropna().values
    
    # PM2.5 điền khuyết (bao gồm dòng nội suy)
    pm25_filled = df_aq["pm25_mean"].dropna().values
    
    # PM2.5 sau xử lý (log-transformed pm25 từ file daily_merged)
    df_merged = pd.read_csv(ROOT / "data/processed/01_daily_merged.csv")
    pm25_processed = df_merged["pm25"].values
    pm25_log = np.log1p(pm25_processed)
    
    # 2. Load Satellite AOD (MAIAC)
    df_maiac_raw = pd.read_csv(ROOT / "data/processed/08_maiac_aod_daily.csv")
    maiac_raw = df_maiac_raw["maiac_aod_mean"].dropna().values
    
    # Imputed & Standardized AOD
    maiac_processed = df_merged["maiac_aod_mean"].dropna().values
    maiac_scaled = (maiac_processed - np.mean(maiac_processed)) / np.std(maiac_processed)
    
    # 3. Load Relative Humidity
    rh_raw = df_aq["relative_humidity_pct_mean"].dropna().values if "relative_humidity_pct_mean" in df_aq.columns else df_merged["relative_humidity_pct_mean"].dropna().values
    rh_scaled = (rh_raw - np.mean(rh_raw)) / np.std(rh_raw)
    
    # Hàm tính toán thống kê mô tả
    def get_row_stats(var_name, stage, vals):
        return {
            "Variable": var_name,
            "Stage": stage,
            "Mean": f"{np.mean(vals):.3f}",
            "Median": f"{np.median(vals):.3f}",
            "Std Dev": f"{np.std(vals):.3f}",
            "Skewness": f"{stats.skew(vals):.3f}",
            "Kurtosis": f"{stats.kurtosis(vals):.3f}"
        }
        
    table_rows = [
        get_row_stats("PM2.5 (ug/m3)", "Raw", pm25_raw),
        get_row_stats("PM2.5 (ug/m3)", "Gap-Filled", pm25_filled),
        get_row_stats("PM2.5 (ug/m3)", "Log-Transformed (log1p)", pm25_log),
        
        get_row_stats("MAIAC AOD", "Raw", maiac_raw),
        get_row_stats("MAIAC AOD", "Imputed", maiac_processed),
        get_row_stats("MAIAC AOD", "Standardized", maiac_scaled),
        
        get_row_stats("Relative Humidity (%)", "Raw", rh_raw),
        get_row_stats("Relative Humidity (%)", "Standardized", rh_scaled),
    ]
    
    df_table = pd.DataFrame(table_rows)
    return (pm25_raw, pm25_log), (maiac_raw, maiac_scaled), df_table

def plot_distributions(pm25_tuple, maiac_tuple):
    raw_pm25, log_pm25 = pm25_tuple
    
    fig, axes = plt.subplots(2, 2, figsize=(14, 11))
    
    # --- Top Left: PDF của PM2.5 Thô (Lệch phải nặng) ---
    sns.histplot(raw_pm25, kde=True, color="#D95F02", ax=axes[0, 0], stat="density", bins=40)
    axes[0, 0].set_title("(a) PDF: Raw PM2.5 Distribution (Right Skewed)", fontweight="bold")
    axes[0, 0].set_xlabel("PM2.5 Concentration (ug/m3)")
    axes[0, 0].set_ylabel("Density")
    
    # --- Top Right: PDF của PM2.5 sau Log-Transform (Dạng chuẩn hóa) ---
    sns.histplot(log_pm25, kde=True, color="#1B9E77", ax=axes[0, 1], stat="density", bins=40)
    axes[0, 1].set_title("(b) PDF: Log-Transformed PM2.5 (Gaussian-like)", fontweight="bold")
    axes[0, 1].set_xlabel("log1p(PM2.5)")
    axes[0, 1].set_ylabel("Density")
    
    # --- Bottom Left: Q-Q Plot của PM2.5 Thô ---
    stats.probplot(raw_pm25, dist="norm", plot=axes[1, 0])
    axes[1, 0].get_lines()[0].set_color("#D95F02")
    axes[1, 0].get_lines()[0].set_alpha(0.6)
    axes[1, 0].get_lines()[1].set_color("black")
    axes[1, 0].set_title("(c) Q-Q Plot: Raw PM2.5 vs. Normal Dist", fontweight="bold")
    axes[1, 0].set_xlabel("Theoretical Quantiles")
    axes[1, 0].set_ylabel("Ordered Values")
    
    # --- Bottom Right: Q-Q Plot của PM2.5 Log-Transform ---
    stats.probplot(log_pm25, dist="norm", plot=axes[1, 1])
    axes[1, 1].get_lines()[0].set_color("#1B9E77")
    axes[1, 1].get_lines()[0].set_alpha(0.6)
    axes[1, 1].get_lines()[1].set_color("black")
    axes[1, 1].set_title("(d) Q-Q Plot: Log-Transformed PM2.5 vs. Normal Dist", fontweight="bold")
    axes[1, 1].set_xlabel("Theoretical Quantiles")
    axes[1, 1].set_ylabel("Ordered Values")
    
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / "distribution_analysis.png", bbox_inches="tight", dpi=300)
    print(f"Saved Figure 4 to: {OUTPUT_DIR / 'distribution_analysis.png'}")
    plt.close()

if __name__ == "__main__":
    pm25, maiac, df_table = calculate_stats_and_data()
    plot_distributions(pm25, maiac)
    
    print("\n=== Table 4: Summary Statistics of Key Variables ===")
    headers = list(df_table.columns)
    md_table = "| " + " | ".join(headers) + " |\n"
    md_table += "| " + " | ".join(["---"] * len(headers)) + " |\n"
    for _, row in df_table.iterrows():
        md_table += "| " + " | ".join(str(val) for val in row) + " |\n"
    print(md_table)
    
    df_table.to_csv(OUTPUT_DIR / "distribution_table.csv", index=False)


Saved Figure 4 to: D:\Bussiness_plan\Multimodal_PM25\outputs\distribution_analysis.png

=== Table 4: Summary Statistics of Key Variables ===
| Variable | Stage | Mean | Median | Std Dev | Skewness | Kurtosis |
| --- | --- | --- | --- | --- | --- | --- |
| PM2.5 (ug/m3) | Raw | 42.047 | 33.472 | 29.408 | 1.807 | 4.393 |
| PM2.5 (ug/m3) | Gap-Filled | 42.828 | 34.235 | 29.519 | 1.648 | 3.663 |
| PM2.5 (ug/m3) | Log-Transformed (log1p) | 3.582 | 3.566 | 0.651 | -0.229 | 0.398 |
| MAIAC AOD | Raw | 0.524 | 0.400 | 0.379 | 2.049 | 6.499 |
| MAIAC AOD | Imputed | 0.539 | 0.395 | 0.430 | 2.205 | 6.570 |
| MAIAC AOD | Standardized | -0.000 | -0.333 | 1.000 | 2.205 | 6.570 |
| Relative Humidity (%) | Raw | 79.568 | 81.396 | 8.493 | -0.889 | 0.635 |
| Relative Humidity (%) | Standardized | 0.000 | 0.215 | 1.000 | -0.889 | 0.635 |



: 